In [7]:
import pandas as pd
import random

# ============================
# Load Dataset
# ============================
df = pd.read_csv("raw/support_tickets.csv")
#E:\projects\Depi\New folder\F1\New folder\Data\raw\support_tickets.csv

# ============================
# Possible Values
# ============================

products = [
    "Web Platform",
    "Mobile App",
    "Payment API",
    "Cloud Dashboard",
    "CRM System",
    "Desktop Client"
]

companies = [
    "Microsoft",
    "Amazon",
    "IBM",
    "Google",
    "Dell",
    "Cisco",
    "Intel",
    "Oracle"
]

languages = [
    "English",
    "Arabic"
]

regions = [
    "USA",
    "UK",
    "Germany",
    "Egypt",
    "Saudi Arabia",
    "UAE"
]

severities = ["Minor", "Major", "Critical"]

knowledge_articles = {
    "billing": "Verify invoice, compare billing history, process refund if needed.",
    "technical": "Restart service, collect logs, investigate server health, deploy hotfix if required.",
    "account": "Verify customer identity, reset credentials, unlock account.",
    "shipping": "Track shipment, contact courier, arrange replacement if necessary.",
    "product": "Provide product documentation and user guide.",
    "general": "Answer customer inquiry using the official knowledge base."
}

root_causes = {
    "billing": "Incorrect billing configuration",
    "technical": "Software bug or infrastructure issue",
    "account": "Authentication failure",
    "shipping": "Courier delay",
    "product": "Missing documentation",
    "general": "Customer information request"
}

# ============================
# Add New Columns
# ============================

df["product"] = [random.choice(products) for _ in range(len(df))]

df["company"] = [random.choice(companies) for _ in range(len(df))]

df["language"] = [random.choice(languages) for _ in range(len(df))]

df["region"] = [random.choice(regions) for _ in range(len(df))]

df["severity"] = [random.choice(severities) for _ in range(len(df))]

# Tags based on category
df["tags"] = df["category"]

# Root Cause
df["root_cause"] = df["category"].map(root_causes)

# Knowledge Article
df["knowledge_article"] = df["category"].map(knowledge_articles)

# Customer Satisfaction
df["customer_satisfaction"] = [
    random.randint(3, 5) if s in ["resolved", "closed"] else None
    for s in df["status"]
]

# ============================
# Conversation
# ============================

df["conversation"] = (
    "Customer: " + df["body"] +
    "\nSupport: " + df["resolution_note"].fillna("Issue is under investigation.")
)

# ============================
# Save
# ============================

output_file = "raw/support_tickets_enhanced.csv"

df.to_csv(output_file, index=False)

print("Dataset saved successfully!")
print(output_file)

Dataset saved successfully!
raw/support_tickets_enhanced.csv


In [9]:
df = pd.read_csv("raw/support_tickets_enhanced.csv")

In [12]:
print(df.columns)

Index(['ticket_id', 'created_at', 'resolved_at', 'channel', 'agent_id',
       'category', 'priority', 'status', 'subject', 'body', 'resolution_note',
       'word_count', 'product', 'company', 'language', 'region', 'severity',
       'tags', 'root_cause', 'knowledge_article', 'customer_satisfaction',
       'conversation'],
      dtype='object')


In [11]:
df.isnull().sum()

ticket_id                  0
created_at                 0
resolved_at              351
channel                    0
agent_id                   0
category                   0
priority                   0
status                     0
subject                    0
body                       0
resolution_note          351
word_count                 0
product                    0
company                    0
language                   0
region                     0
severity                   0
tags                       0
root_cause                 0
knowledge_article          0
customer_satisfaction    351
conversation               0
dtype: int64

In [13]:
df["resolved_at"] = df["resolved_at"].fillna("Not Resolved")

df["resolution_note"] = df["resolution_note"].fillna("Pending Resolution")

df["customer_satisfaction"] = df["customer_satisfaction"].fillna(0)

In [14]:
print(df.isnull().sum())

ticket_id                0
created_at               0
resolved_at              0
channel                  0
agent_id                 0
category                 0
priority                 0
status                   0
subject                  0
body                     0
resolution_note          0
word_count               0
product                  0
company                  0
language                 0
region                   0
severity                 0
tags                     0
root_cause               0
knowledge_article        0
customer_satisfaction    0
conversation             0
dtype: int64


In [16]:
df.duplicated().sum()

np.int64(0)

In [17]:
df["created_at"] = pd.to_datetime(df["created_at"])

df["resolved_at"] = pd.to_datetime(
    df["resolved_at"],
    errors="coerce"
)

In [18]:
df["priority"] = df["priority"].astype("category")
df["status"] = df["status"].astype("category")
df["category"] = df["category"].astype("category")

In [20]:
print(df.columns.tolist())

['ticket_id', 'created_at', 'resolved_at', 'channel', 'agent_id', 'category', 'priority', 'status', 'subject', 'body', 'resolution_note', 'word_count', 'product', 'company', 'language', 'region', 'severity', 'tags', 'root_cause', 'knowledge_article', 'customer_satisfaction', 'conversation']


In [21]:
df["text"] = (
    df["subject"] + " " +
    df["body"] + " " +
    df["resolution_note"].fillna("")
)

In [22]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df["clean_text"] = df["text"].apply(clean_text)

In [23]:
df.to_csv("raw/support_tickets_preprocessed.csv", index=False)